# behav_utils — Example Workflow
## Demonstrating the three-level convention with synthetic data

This notebook shows the full pipeline without any real data:
1. **Generate** synthetic sessions with known parameters
2. **Filter** trials
3. **Analyse** (low-level and session-level)
4. **Plot** (from result dicts only)
5. **Compare** two conditions


## Setup


In [ ]:
from pathlib import Path
import os, sys

_NOTEBOOK_DIR = Path(os.path.abspath(''))
_BEHAV_UTIL_DIR= _NOTEBOOK_DIR.parent
_PROJECT_ROOT = _BEHAV_UTIL_DIR.parent
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from behav_utils import (
    # Synthetic data
    generate_synthetic_animal, sample_stimuli,
    # Filtering
    filter_trials, pool_arrays,
    # Low-level analysis (arrays)
    fit_psychometric, compute_update_matrix, compare_conditions, cumulative_gaussian,
    # Session-level analysis (sessions -> result dicts)
    compute_psychometric, compute_um, compute_trajectory,
    compute_comparison, compute_session_raster,
    # Plotting (result dicts -> axes)
    plot_psychometric, plot_um, plot_trajectory,
    plot_comparison, plot_session_raster,
    # Styles
    PALETTE, apply_style,
)
from behav_utils.data.synthetic import noisy_psychometric_simulator

apply_style()
print('All imports successful.')


## 1. Generate synthetic data

We create two synthetic animals with different psychometric parameters.
- **Animal A**: steep psychometric (sigma=0.15), low lapse
- **Animal B**: shallow psychometric (sigma=0.40), high lapse

`noisy_psychometric_simulator` signature: `(stimuli, categories, rng, sigma=0.3, lapse=0.05)`.
Parameters are passed via `simulator_kwargs`, not as direct arguments.


In [ ]:
animal_a, info_a = generate_synthetic_animal(
    animal_id='SynA',
    n_sessions=10,
    trials_per_session=150,
    simulator=noisy_psychometric_simulator,
    simulator_kwargs={'sigma': 0.15, 'lapse': 0.02},
    seed=42,
)

animal_b, info_b = generate_synthetic_animal(
    animal_id='SynB',
    n_sessions=10,
    trials_per_session=150,
    simulator=noisy_psychometric_simulator,
    simulator_kwargs={'sigma': 0.40, 'lapse': 0.10},
    seed=99,
)

print(f'Animal A: {animal_a.n_sessions} sessions, sigma=0.15, lapse=0.02')
print(f'Animal B: {animal_b.n_sessions} sessions, sigma=0.40, lapse=0.10')


## 2. Low-level functions (raw arrays)

These work with any arrays — model output, synthetic data, numpy.
No SessionData required.


In [ ]:
# Generate raw arrays
rng = np.random.default_rng(42)
stimuli, categories = sample_stimuli(500, distribution='uniform', rng=rng)
choices = noisy_psychometric_simulator(stimuli, categories, rng, sigma=0.20, lapse=0.03)

# Low-level psychometric fit
params = fit_psychometric(stimuli, choices)
print('fit_psychometric result:')
for k in ('mu', 'sigma', 'lapse_low', 'lapse_high', 'success'):
    v = params.get(k)
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')


In [ ]:
# Low-level update matrix
um, conditional, info = compute_update_matrix(stimuli, choices, categories)
print(f'Update matrix shape: {um.shape}')
print(f'Total trials used: {info["total_trials"]}')


In [ ]:
# Low-level comparison: two independent arrays
rng2 = np.random.default_rng(99)
stimuli_2, categories_2 = sample_stimuli(500, distribution='uniform', rng=rng2)
choices_2 = noisy_psychometric_simulator(stimuli_2, categories_2, rng2, sigma=0.40, lapse=0.10)

comp = compare_conditions(
    stimuli, choices, categories,
    stimuli_2, choices_2, categories_2,
    n_permutations=500, n_bootstrap=500,
    label_a='Sharp', label_b='Shallow',
)
print(f'PSE diff: {comp["diffs"]["pse"]:.4f}')
print(f'Permutation p (PSE): {comp["perm_p"]["pse"]:.4f}')
print(f'UM RMSE: {comp["um_rmse"]:.4f}')


## 3. Session-level analysis (sessions → result dicts)

These take pre-filtered `List[SessionData]` and return structured dicts
ready for the corresponding `plot_` function.


In [ ]:
# Filter trials (synthetic data has no opto, so this just excludes aborts)
sessions_a = list(animal_a.sessions)
clean_a = filter_trials(sessions_a)

sessions_b = list(animal_b.sessions)
clean_b = filter_trials(sessions_b)

print(f'Animal A: {len(clean_a)} sessions after filtering')
print(f'Animal B: {len(clean_b)} sessions after filtering')


In [ ]:
# compute_psychometric — pooled mode
psych_a = compute_psychometric(clean_a, mode='pooled', n_bootstrap=200)
psych_b = compute_psychometric(clean_b, mode='pooled', n_bootstrap=200)

print(f"Animal A pooled PSE: {psych_a['params'].get('mu', float('nan')):.4f}")
print(f"Animal B pooled PSE: {psych_b['params'].get('mu', float('nan')):.4f}")
print(f"\nResult keys: {sorted(psych_a.keys())}")


In [ ]:
# compute_psychometric — overlay and session_mean modes
psych_overlay = compute_psychometric(clean_a, mode='overlay')
psych_mean = compute_psychometric(clean_a, mode='session_mean', n_bootstrap=100)

print(f"Overlay: {psych_overlay['n_sessions']} session fits")
print(f"Session mean: {psych_mean['n_sessions']} sessions averaged")


In [ ]:
# compute_um
um_a = compute_um(clean_a)
um_b = compute_um(clean_b)

print(f"UM result keys: {sorted(um_a.keys())}")
print(f"UM shape: {um_a['um'].shape}")
print(f"Sessions pooled: {um_a['n_sessions']}")


In [ ]:
# compute_trajectory
traj_a = compute_trajectory(clean_a, stat_names=['accuracy', 'pse'])
traj_b = compute_trajectory(clean_b, stat_names=['accuracy', 'pse'])

print(f"Trajectory keys: {sorted(traj_a.keys())}")
print(f"Stats computed: {traj_a['stat_names']}")
print(f"Accuracy values: {np.round(traj_a['values']['accuracy'], 3)}")


In [ ]:
# compute_comparison (session-level)
comp_result = compute_comparison(
    clean_a, clean_b,
    n_permutations=500, n_bootstrap=500,
    label_a='Animal A', label_b='Animal B',
)

print(f"PSE diff (A - B): {comp_result['diffs']['pse']:.4f}")
print(f"Accuracy diff: {comp_result['diffs']['accuracy']:.4f}")
print(f"Sessions: {comp_result['n_sessions_a']}A vs {comp_result['n_sessions_b']}B")


In [ ]:
# compute_session_raster
raster = compute_session_raster(clean_a[0])

print(f"Raster keys: {sorted(raster.keys())}")
print(f"Trials: {raster['n_trials']}")
print(f"Correct: {raster['correct'].sum()}/{raster['n_trials']}")


## 4. Plotting (result dicts → axes)

Every plot function takes a result dict. No computation inside.
User controls layout with `plt.subplots()`.


In [ ]:
# Psychometric: pooled, two animals overlaid
fig, ax = plt.subplots(1, 1, figsize=(5, 4))
plot_psychometric(psych_a, ax=ax, color=PALETTE[0], label='Animal A (sigma=0.15)')
plot_psychometric(psych_b, ax=ax, color=PALETTE[1], label='Animal B (sigma=0.40)')
ax.legend(fontsize=9)
ax.set_title('Pooled Psychometric Comparison')
plt.tight_layout()
plt.show()


In [ ]:
# Psychometric: three modes side by side
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

plot_psychometric(psych_a, ax=axes[0], color=PALETTE[0], title='Pooled')
plot_psychometric(psych_overlay, ax=axes[1], title='Overlay (per-session)')
plot_psychometric(psych_mean, ax=axes[2], color=PALETTE[2], title='Session Mean +/- SEM')

plt.tight_layout()
plt.show()


In [ ]:
# Update matrices
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_um(um_a, ax=axes[0], title='Animal A')
plot_um(um_b, ax=axes[1], title='Animal B')
plt.tight_layout()
plt.show()


In [ ]:
# UM difference (raw ndarray arithmetic, then plot)
diff = um_a['um'] - um_b['um']
fig, ax = plt.subplots(1, 1, figsize=(5, 4))
plot_um(diff, ax=ax, title='A - B (UM difference)')
plt.tight_layout()
plt.show()


In [ ]:
# Trajectory
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

plot_trajectory(traj_a, 'accuracy', ax=axes[0], color=PALETTE[0], label='Animal A')
plot_trajectory(traj_b, 'accuracy', ax=axes[0], color=PALETTE[1], label='Animal B')
axes[0].legend(fontsize=9)
axes[0].set_title('Accuracy Trajectory')

plot_trajectory(traj_a, 'pse', ax=axes[1], color=PALETTE[0], label='Animal A')
plot_trajectory(traj_b, 'pse', ax=axes[1], color=PALETTE[1], label='Animal B')
axes[1].legend(fontsize=9)
axes[1].set_title('PSE Trajectory')

plt.tight_layout()
plt.show()


In [ ]:
# Comparison plots
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_comparison(comp_result, ax=axes[0], metric='psychometric', title='Psychometric')
plot_comparison(comp_result, ax=axes[1], metric='accuracy', title='Accuracy')
plt.tight_layout()
plt.show()


In [ ]:
# Comparison: UM mode (creates its own 3-panel figure)
fig, axes = plot_comparison(comp_result, metric='um', title='UM Comparison')
plt.tight_layout()
plt.show()


In [ ]:
# Session raster
fig, ax = plot_session_raster(raster, window=20)
plt.tight_layout()
plt.show()


## 5. Using low-level functions with model predictions

Low-level functions accept raw arrays, so you can use them with
model-generated data, posterior predictive checks, or any custom source.


In [ ]:
# Generate theoretical psychometric curves directly
x = np.linspace(-1, 1, 200)
y_sharp = cumulative_gaussian(x, mu=0.0, sigma=0.15, lapse_low=0.02, lapse_high=0.02)
y_shallow = cumulative_gaussian(x, mu=0.0, sigma=0.40, lapse_low=0.10, lapse_high=0.10)

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(x, y_sharp, color=PALETTE[0], lw=2, label='Sharp (sigma=0.15)')
ax.plot(x, y_shallow, color=PALETTE[1], lw=2, label='Shallow (sigma=0.40)')
ax.axhline(0.5, ls='--', color='grey', alpha=0.3)
ax.axvline(0.0, ls='--', color='grey', alpha=0.3)
ax.set_xlabel('Stimulus')
ax.set_ylabel('P(choose B)')
ax.set_title('Model Predictions (low-level cumulative_gaussian)')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# compute_psychometric also accepts raw (stimuli, choices) tuples
# — useful for model-generated arrays without SessionData
result_raw = compute_psychometric((stimuli, choices), mode='pooled')
print(f"Fit from raw tuple: PSE = {result_raw['params'].get('mu', float('nan')):.4f}")

fig, ax = plt.subplots(figsize=(5, 4))
plot_psychometric(result_raw, ax=ax, color=PALETTE[0], title='From raw arrays')
plt.tight_layout()
plt.show()


## 6. Learning trajectory (per-session varying parameters)

`generate_synthetic_animal` supports `per_session_simulator_kwargs`
to model learning trajectories with changing parameters.


In [ ]:
n_sess = 20
animal_learn, _ = generate_synthetic_animal(
    animal_id='LEARN01',
    n_sessions=n_sess,
    trials_per_session=200,
    simulator=noisy_psychometric_simulator,
    per_session_simulator_kwargs=[
        {'sigma': 0.8 - i * 0.03, 'lapse': max(0.01, 0.15 - i * 0.006)}
        for i in range(n_sess)
    ],
    seed=456,
)

clean_learn = filter_trials(list(animal_learn.sessions))
traj_learn = compute_trajectory(clean_learn, ['accuracy', 'pse', 'slope'])

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for i, stat in enumerate(['accuracy', 'pse', 'slope']):
    plot_trajectory(traj_learn, stat, ax=axes[i], color=PALETTE[2])
    axes[i].set_title(stat)
plt.suptitle('Learning Trajectory (sigma decreasing across sessions)', y=1.02)
plt.tight_layout()
plt.show()


## Summary

| Level | Function | Input | Output |
|:------|:---------|:------|:-------|
| **Low-level** | `fit_psychometric(stim, ch)` | Raw arrays | Params dict |
| **Low-level** | `compute_update_matrix(s, ch, cat)` | Raw arrays | (um, cond, info) |
| **Low-level** | `compare_conditions(...)` | Raw arrays x 2 | Comparison dict |
| **Session-level** | `compute_psychometric(sessions)` | List[SessionData] | Result dict |
| **Session-level** | `compute_um(sessions)` | List[SessionData] | Result dict |
| **Session-level** | `compute_trajectory(sessions, stats)` | List[SessionData] | Result dict |
| **Session-level** | `compute_comparison(a, b)` | List x 2 | Result dict |
| **Session-level** | `compute_session_raster(session)` | SessionData | Result dict |
| **Plotting** | `plot_psychometric(result)` | Result dict | (fig, ax) |
| **Plotting** | `plot_um(result)` | Result dict | (fig, ax) |
| **Plotting** | `plot_trajectory(result, stat)` | Result dict | (fig, ax) |
| **Plotting** | `plot_comparison(result, metric)` | Result dict | (fig, ax) |
| **Plotting** | `plot_session_raster(result)` | Result dict | (fig, ax) |

Analysis returns results. Plotting draws results. No exceptions.
